In [13]:
# scripts/make_dashboard.py
"""
Dashboard builder (static HTML, no server)
==========================================
Inputs (repo-relative):
- data/processed/data_clean.csv

Outputs:
- docs/dashboard.html                   (interactive Plotly dashboard)
- outputs/dashboard_country_agg.csv     (country-level aggregates)
- outputs/dashboard_wg_totals.csv       (WG totals, if WG columns exist)
- outputs/itc_summary.csv               (ITC vs Non-ITC rollups, if ITC column exists)
- docs/downloads/country_agg.csv
- docs/downloads/wg_totals.csv
- docs/downloads/itc_summary.csv

Notes
-----
- Static Plotly HTML that works on GitHub Pages (no server).
- Clean, consistent theme; **All Data** explorers to cover every country.
"""

from __future__ import annotations
from pathlib import Path
import sys, re, shutil
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# --------------------------- repo root detection --------------------------- #
def _find_repo_root() -> Path:
    try:
        here = Path(__file__).resolve()
        return here.parent.parent  # …/scripts → repo root
    except NameError:
        cwd = Path.cwd().resolve()
        if (cwd / "data").is_dir() and (cwd / "scripts").is_dir():
            return cwd
        if cwd.name == "scripts" and (cwd.parent / "data").is_dir():
            return cwd.parent
        cur = cwd
        for _ in range(5):
            if (cur / ".git").is_dir() or ((cur / "data").is_dir() and (cur / "scripts").is_dir()):
                return cur
            cur = cur.parent
        return cwd

ROOT      = _find_repo_root()
CSV_PATH  = ROOT / "data" / "processed" / "data_clean.csv"
DOCS_DIR  = ROOT / "docs"
OUT_DIR   = ROOT / "outputs"
DL_DIR    = DOCS_DIR / "downloads"
DOCS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
DL_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root →", ROOT)
if not CSV_PATH.exists():
    sys.exit(f"ERROR: {CSV_PATH.relative_to(ROOT)} not found. Run scripts/make_eda.py first.")

# ------------------------------ theme ------------------------------------- #
pio.templates.default = "plotly_white"
px.defaults.template = "plotly_white"
px.defaults.width = None
px.defaults.height = 420

ACCENT = "#1F5FCC"
ITC_GREEN = "#147D64"
NONITC_GREY = "#6B7280"

# ------------------------------ helpers ----------------------------------- #
def yn_to_bool(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
         .str.strip().str.lower()
         .map({"y": True, "yes": True, "member": True, "1": True, "true": True, "t": True, "x": True,
               "n": False, "no": False, "0": False, "false": False, "" : False})
         .fillna(False)
    )

DIGIT_RE = re.compile(r"\d+")
def wg_pretty(name: str) -> str:
    m = DIGIT_RE.search(str(name))
    return f"WG {m.group(0)}" if m else str(name)

def first_js(fig: go.Figure) -> str:
    return pio.to_html(fig, include_plotlyjs=True, full_html=False, default_width="100%", default_height="100%")

def later_js(fig: go.Figure) -> str:
    return pio.to_html(fig, include_plotlyjs=False, full_html=False, default_width="100%", default_height="100%")

def safe_div(fig: go.Figure | None, first: bool=False) -> str:
    if fig is None:
        return ""
    return first_js(fig) if first else later_js(fig)

def tweak(fig, title=None, height=None):
    if title: fig.update_layout(title=title)
    if height: fig.update_layout(height=height)
    fig.update_layout(margin=dict(l=10, r=10, t=60, b=10), hovermode="closest")
    return fig

# ------------------------------ load/clean -------------------------------- #
df = pd.read_csv(CSV_PATH, low_memory=False)
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].astype(str).str.replace("\u00A0", " ", regex=False).str.strip()

country_col = "country_clean" if "country_clean" in df.columns else ("country" if "country" in df.columns else None)
if not country_col:
    sys.exit("ERROR: no 'country_clean' or 'country' column in data_clean.csv")

mc_col   = "mc_member"  if "mc_member"  in df.columns else None
core_col = "core_group" if "core_group" in df.columns else None
wg_cols  = [c for c in df.columns if re.match(r"(?i)^wg\d", c)]
wg_cols  = sorted(wg_cols, key=lambda x: int(re.findall(r"\d+", x)[0]) if re.findall(r"\d+", x) else 99)
wg_member_col = "wg_member" if "wg_member" in df.columns else None

if mc_col and df[mc_col].dtype != bool: df[mc_col] = yn_to_bool(df[mc_col])
else:
    df["mc_member"] = False; mc_col = "mc_member"

if core_col and df[core_col].dtype != bool: df[core_col] = yn_to_bool(df[core_col])
else:
    df["core_group"] = False; core_col = "core_group"

for c in wg_cols:
    if df[c].dtype != bool: df[c] = yn_to_bool(df[c])

df["any_wg"] = df[wg_cols].any(axis=1) if wg_cols else False
if wg_member_col: df["any_wg"] = df["any_wg"] | yn_to_bool(df[wg_member_col])

have_itc = "itc_countries" in df.columns
if have_itc:
    itc_norm = df["itc_countries"].astype(str).str.strip().str.lower()
    df["itc_bool"] = itc_norm.isin(["yes","y","true","1"])
else:
    df["itc_bool"] = False

# ------------------------------ aggregates -------------------------------- #
agg_parts = {
    "Country":      (country_col, "first"),
    "Total":        (country_col, "size"),
    "MC":           (mc_col, "sum"),
    "Core":         (core_col, "sum"),
    "Any WG":       ("any_wg", "sum"),
    "ITC participants": ("itc_bool", "sum"),
}
for c in wg_cols: agg_parts[c] = (c, "sum")

country_agg = df.groupby(country_col, as_index=False).agg(**agg_parts)

for c in ["Total", "MC", "Core", "Any WG", "ITC participants"] + wg_cols:
    if c in country_agg.columns:
        country_agg[c] = country_agg[c].fillna(0).astype(int)

country_itc_flag = (
    df.groupby(country_col)["itc_bool"]
      .any()
      .reindex(country_agg["Country"])
      .fillna(False)
      .astype(bool)
      .values
)
country_agg["ITC country flag"] = np.where(country_itc_flag, 1, 0).astype(int)
country_agg["ITC group"] = np.where(country_agg["ITC country flag"] == 1, "ITC", "Non-ITC")
country_agg["AnyWG_rate"] = np.where(country_agg["Total"]>0, country_agg["Any WG"]/country_agg["Total"], 0.0)
country_agg["ITC_share"]  = np.where(country_agg["Total"]>0, country_agg["ITC participants"]/country_agg["Total"], 0.0)

kpi_total_people  = int(df.shape[0])
kpi_countries     = int(country_agg["Country"].nunique())
kpi_mc_total      = int(country_agg["MC"].sum())
kpi_core_total    = int(country_agg["Core"].sum())
kpi_any_wg_total  = int(country_agg["Any WG"].sum())
kpi_itc_people    = int(df["itc_bool"].sum()) if have_itc else 0
kpi_itc_countries = int((country_agg["ITC country flag"] == 1).sum()) if have_itc else 0

if wg_cols:
    wg_totals = country_agg[wg_cols].sum().rename_axis("wg").reset_index(name="count")
else:
    wg_totals = pd.DataFrame(columns=["wg", "count"])

if have_itc:
    itc_row = df.groupby("itc_bool").agg(
        Participants=("any_wg","size"),
        MC=("mc_member","sum"),
        Core=("core_group","sum"),
        Any_WG=("any_wg","sum"),
        **({c:(c,"sum") for c in wg_cols})
    ).reset_index()
    itc_row["ITC"] = np.where(itc_row["itc_bool"], "ITC", "Non-ITC")
    itc_row["Any_WG_rate"] = np.where(itc_row["Participants"]>0, itc_row["Any_WG"]/itc_row["Participants"], 0.0)
    itc_summary = itc_row[["ITC","Participants","MC","Core","Any_WG","Any_WG_rate"]].rename(columns={"ITC":"bucket"})
else:
    itc_row = pd.DataFrame(); itc_summary = pd.DataFrame()

# ------------------------------ exports ----------------------------------- #
out_country_csv = OUT_DIR / "dashboard_country_agg.csv"
country_agg.to_csv(out_country_csv, index=False, encoding="utf-8-sig")
shutil.copyfile(out_country_csv, DL_DIR / "country_agg.csv")
print("Saved →", out_country_csv.relative_to(ROOT))
print("Saved →", (DL_DIR / "country_agg.csv").relative_to(ROOT))

if not wg_totals.empty:
    out_wg_csv = OUT_DIR / "dashboard_wg_totals.csv"
    wg_totals.to_csv(out_wg_csv, index=False, encoding="utf-8-sig")
    shutil.copyfile(out_wg_csv, DL_DIR / "wg_totals.csv")
    print("Saved →", out_wg_csv.relative_to(ROOT))
    print("Saved →", (DL_DIR / "wg_totals.csv").relative_to(ROOT))

if have_itc:
    out_itc_csv = OUT_DIR / "itc_summary.csv"
    itc_summary.to_csv(out_itc_csv, index=False, encoding="utf-8-sig")
    shutil.copyfile(out_itc_csv, DL_DIR / "itc_summary.csv")
    print("Saved →", out_itc_csv.relative_to(ROOT))
    print("Saved →", (DL_DIR / "itc_summary.csv").relative_to(ROOT))

# ------------------------------ figures — overview ------------------------- #
cd_map = {"ITC": ITC_GREEN, "Non-ITC": NONITC_GREY}

# Top countries (quick read)
topN = 15
top_countries = country_agg.sort_values("Total", ascending=False).head(topN)
fig_top_total = px.bar(
    top_countries, x="Total", y="Country", color="ITC group",
    color_discrete_map=cd_map, orientation="h",
    title=f"Top {min(topN, len(country_agg))} Countries by Total People"
)
fig_top_total.update_layout(yaxis={"categoryorder": "total ascending"})
tweak(fig_top_total)

# Treemap grouped by ITC → Country
fig_treemap_group = px.treemap(
    country_agg.sort_values("Total", ascending=False),
    path=["ITC group", "Country"], values="Total",
    color="ITC group", color_discrete_map=cd_map,
    title="Treemap: Total people by country (grouped by ITC vs Non-ITC)"
)
tweak(fig_treemap_group)

# Bubble: MC vs Core (size=Total, color=ITC group)
fig_bubble_mc_core = px.scatter(
    country_agg, x="MC", y="Core", size="Total", color="ITC group",
    color_discrete_map=cd_map, hover_name="Country",
    title="MC vs Core by country (size = Total; color = ITC group)"
)
tweak(fig_bubble_mc_core)

# Pareto (all countries)
pareto = country_agg.sort_values("Total", ascending=False).copy()
pareto["cum_total"] = pareto["Total"].cumsum()
pareto["cum_pct"]   = pareto["cum_total"] / pareto["Total"].sum() * 100
fig_pareto = go.Figure()
fig_pareto.add_bar(x=pareto["Country"], y=pareto["Total"], name="Total", marker_color=ACCENT)
fig_pareto.add_trace(go.Scatter(x=pareto["Country"], y=pareto["cum_pct"], mode="lines+markers",
                                name="Cumulative %", yaxis="y2"))
fig_pareto.update_layout(
    title="Pareto of country totals (all countries)",
    yaxis=dict(title="Total"),
    yaxis2=dict(title="Cumulative %", overlaying="y", side="right", range=[0, 100]),
    xaxis=dict(tickangle=-45)
)
tweak(fig_pareto, height=520)

# Any WG share (row level)
has_wg = int((df["any_wg"] == True).sum())   # noqa: E712
no_wg  = int((df["any_wg"] == False).sum())  # noqa: E712
fig_anywg_pie = px.pie(
    pd.DataFrame({"label": ["Any WG", "No WG"], "count": [has_wg, no_wg]}),
    names="label", values="count", hole=0.4, title="Participants with Any WG membership"
)
fig_anywg_pie.update_traces(textposition="inside", textinfo="percent+label")
tweak(fig_anywg_pie)

# ------------------------------ figures — ITC focus ------------------------ #
if have_itc:
    itc_counts = df["itc_bool"].value_counts(dropna=False).rename(index={True:"ITC", False:"Non-ITC"})
    fig_itc_donut = px.pie(
        pd.DataFrame({"label": itc_counts.index, "count": itc_counts.values}),
        names="label", values="count", hole=0.45, title="Participants: ITC vs Non-ITC",
        color="label", color_discrete_map=cd_map
    )
    fig_itc_donut.update_traces(textposition="inside", textinfo="percent+label")
    tweak(fig_itc_donut)

    tall = itc_row.melt(id_vars=["ITC"], value_vars=["MC","Core"], var_name="Role", value_name="Count")
    fig_itc_roles = px.bar(
        tall, x="Role", y="Count", color="ITC", barmode="group",
        color_discrete_map=cd_map, title="MC and Core totals — ITC vs Non-ITC (participants)"
    )
    tweak(fig_itc_roles)

    rate_df = itc_row[["ITC","Any_WG_rate"]]
    fig_itc_anywg_rate = px.bar(
        rate_df, x="ITC", y="Any_WG_rate", color="ITC", color_discrete_map=cd_map,
        title="Any-WG membership rate — ITC vs Non-ITC", text="Any_WG_rate"
    )
    fig_itc_anywg_rate.update_traces(texttemplate="%{text:.1%}")
    fig_itc_anywg_rate.update_yaxes(tickformat=".0%")
    tweak(fig_itc_anywg_rate)

    fig_itc_country_box = px.box(
        country_agg, x="ITC group", y="Total", color="ITC group",
        color_discrete_map=cd_map, points="all",
        title="Country totals — distribution (ITC vs Non-ITC)"
    )
    tweak(fig_itc_country_box)

    top_itc = country_agg[country_agg["ITC country flag"] == 1].sort_values("Total", ascending=False).head(15)
    fig_top_itc = px.bar(
        top_itc, x="Total", y="Country", orientation="h",
        color="ITC group", color_discrete_map=cd_map,
        title=f"Top {min(15, len(top_itc))} ITC countries by Total people"
    )
    fig_top_itc.update_layout(yaxis={"categoryorder": "total ascending"})
    tweak(fig_top_itc)

    fig_scatter_itc = px.scatter(
        country_agg, x="MC", y="Core", size="Total", color="ITC group",
        color_discrete_map=cd_map, hover_name="Country",
        title="MC vs Core — colored by ITC vs Non-ITC"
    )
    tweak(fig_scatter_itc)

    if wg_cols:
        rows = []
        for label, subset in df.groupby(df["itc_bool"].map({True:"ITC", False:"Non-ITC"})):
            sums = subset[wg_cols].astype(int).sum()
            tot  = int(sums.sum()) if int(sums.sum()) > 0 else 1
            for c in wg_cols:
                rows.append({"ITC": label, "WG": wg_pretty(c), "Share": sums[c] / tot})
        comp = pd.DataFrame(rows)
        fig_itc_wg_share = px.bar(
            comp, x="Share", y="ITC", color="WG", orientation="h",
            barmode="stack", title="WG composition (share) — ITC vs Non-ITC (100% stacked)"
        )
        fig_itc_wg_share.update_xaxes(tickformat=".0%")
        tweak(fig_itc_wg_share, height=460)
    else:
        fig_itc_wg_share = None
else:
    fig_itc_donut = None; fig_itc_roles = None; fig_itc_anywg_rate = None
    fig_itc_country_box = None; fig_top_itc = None; fig_scatter_itc = None; fig_itc_wg_share = None

# ------------------------------ figures — WG pack -------------------------- #
if wg_cols:
    top10 = country_agg.sort_values("Total", ascending=False).head(10).copy()
    long_wg = top10[["Country"] + wg_cols].melt(id_vars="Country", var_name="WG", value_name="Count")
    long_wg["WG"] = long_wg["WG"].map(wg_pretty)
    fig_wg_grouped = px.bar(
        long_wg, x="Count", y="Country", color="WG",
        orientation="h", barmode="group",
        title="Working Group membership — Top 10 countries (grouped)"
    )
    fig_wg_grouped.update_layout(yaxis={"categoryorder": "total ascending"})
    tweak(fig_wg_grouped)

    top10_norm = country_agg.sort_values("Total", ascending=False).head(10).copy()
    totals = top10_norm[wg_cols].astype(int).sum(axis=1).replace(0, 1)
    shares = top10_norm[wg_cols].astype(int).div(totals, axis=0)
    long_wg_pct = pd.concat([top10_norm[["Country"]], shares], axis=1)
    long_wg_pct = long_wg_pct.melt(id_vars="Country", var_name="WG", value_name="Share")
    long_wg_pct["WG"] = long_wg_pct["WG"].map(wg_pretty)
    fig_wg_100 = px.bar(
        long_wg_pct, x="Share", y="Country", color="WG",
        orientation="h", barmode="stack",
        title="WG composition (share) — Top 10 countries (100% stacked)"
    )
    fig_wg_100.update_layout(yaxis={"categoryorder": "total ascending"})
    fig_wg_100.update_xaxes(tickformat=".0%")
    tweak(fig_wg_100)

    corr = None
    if len(wg_cols) >= 2:
        corr = df[wg_cols].astype(int).corr()
        pretty_cols = [wg_pretty(c) for c in corr.columns]
        fig_wg_corr = go.Figure(data=go.Heatmap(
            z=corr.values, x=pretty_cols, y=pretty_cols, zmin=-1, zmax=1, colorscale="RdBu"
        ))
        tweak(fig_wg_corr, "Correlation of WG memberships (row-level)", height=520)
    else:
        fig_wg_corr = None

    # ---- All Data: Country × WG heatmap (all countries) ----
    mat_all = country_agg[["Country"] + wg_cols].set_index("Country").astype(int)
    # sort by total WG membership per country
    mat_all = mat_all.loc[mat_all.sum(axis=1).sort_values(ascending=False).index]
    all_y = mat_all.index.tolist()
    fig_country_wg_heat_all = go.Figure(data=go.Heatmap(
        z=mat_all.values, x=[wg_pretty(c) for c in mat_all.columns], y=all_y, colorscale="Blues"
    ))
    big_h = min(1200, 22*len(all_y) + 120)  # tall, but we’ll put it in a scroll box
    tweak(fig_country_wg_heat_all, "Country × WG membership (all countries)", height=big_h)
else:
    fig_wg_grouped = None; fig_wg_100 = None; fig_wg_corr = None; fig_country_wg_heat_all = None

# ------------------------------ figures — ALL DATA ------------------------- #
# Country Explorer (ALL countries) with metric dropdown
metrics = ["Total", "MC", "Core", "Any WG"] + wg_cols
metric_pretty = {m: (m if m in ["Total","MC","Core","Any WG"] else wg_pretty(m)) for m in metrics}

def build_country_explorer():
    # precompute arrays per metric (sorted desc)
    arrays = {}
    for m in metrics:
        tmp = country_agg[["Country", m, "ITC group"]].copy()
        tmp = tmp.sort_values(m, ascending=False)
        arrays[m] = {
            "y": tmp["Country"].tolist(),
            "x": tmp[m].tolist(),
            "colors": [ITC_GREEN if g=="ITC" else NONITC_GREY for g in tmp["ITC group"]]
        }
    init = metrics[0]
    height = max(560, 18*len(country_agg) + 120)

    fig = go.Figure(data=[go.Bar(
        x=arrays[init]["x"], y=arrays[init]["y"], orientation="h",
        marker=dict(color=arrays[init]["colors"]), hovertemplate="%{y}: %{x}<extra></extra>"
    )])
    fig.update_layout(
        title=f"Country Explorer — {metric_pretty[init]} (all countries)",
        yaxis=dict(categoryorder="array", categoryarray=arrays[init]["y"])
    )
    buttons = []
    for m in metrics:
        buttons.append(dict(
            label=metric_pretty[m],
            method="update",
            args=[
                {
                    "x": [arrays[m]["x"]],
                    "y": [arrays[m]["y"]],
                    "marker": [dict(color=arrays[m]["colors"])],
                },
                {
                    "title": f"Country Explorer — {metric_pretty[m]} (all countries)",
                    "yaxis": {"categoryorder":"array", "categoryarray": arrays[m]["y"]}
                }
            ]
        ))
    fig.update_layout(
        updatemenus=[dict(type="dropdown", x=0.99, xanchor="right", y=1.08, yanchor="top",
                          showactive=True, buttons=buttons)]
    )
    tweak(fig, height=height)
    return fig

fig_country_explorer = build_country_explorer()

# ITC share by country (all countries)
if have_itc:
    itc_share_sorted = country_agg.sort_values("ITC_share", ascending=False)
    height_share = max(560, 18*len(itc_share_sorted) + 120)
    fig_itc_share = px.bar(
        itc_share_sorted, x="ITC_share", y="Country",
        orientation="h", title="ITC share by country (ITC participants ÷ Total)",
    )
    fig_itc_share.update_traces(hovertemplate="%{y}: %{x:.1%}<extra></extra>")
    fig_itc_share.update_xaxes(tickformat=".0%")
    tweak(fig_itc_share, height=height_share)
else:
    fig_itc_share = None

# ------------------------------ assemble HTML ------------------------------ #
PLOTLY_CDN = '<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>'

downloads_links = ["<a href='downloads/country_agg.csv'>country_agg.csv</a>"]
if not wg_totals.empty: downloads_links.append("<a href='downloads/wg_totals.csv'>wg_totals.csv</a>")
if have_itc: downloads_links.append("<a href='downloads/itc_summary.csv'>itc_summary.csv</a>")
downloads_html = " • ".join(downloads_links)

first_chart_html = safe_div(fig_top_total, first=True)

grid_html_overview = f"""
<section id="overview" class="grid">
  <div class="card">{first_chart_html}</div>
  <div class="card">{safe_div(fig_anywg_pie)}</div>
  <div class="card">{safe_div(fig_treemap_group)}</div>
  <div class="card">{safe_div(fig_bubble_mc_core)}</div>
  <div class="card span-2">{safe_div(fig_pareto)}</div>
</section>
"""

grid_html_itc = f"""
<section id="itc" class="grid">
  {"<div class='card'>" + safe_div(fig_itc_donut)       + "</div>" if fig_itc_donut else ""}
  {"<div class='card'>" + safe_div(fig_itc_roles)       + "</div>" if fig_itc_roles else ""}
  {"<div class='card'>" + safe_div(fig_itc_anywg_rate)  + "</div>" if fig_itc_anywg_rate else ""}
  {"<div class='card'>" + safe_div(fig_itc_country_box) + "</div>" if fig_itc_country_box else ""}
  {"<div class='card'>" + safe_div(fig_top_itc)         + "</div>" if fig_top_itc else ""}
  {"<div class='card span-2'>" + safe_div(fig_scatter_itc) + "</div>" if fig_scatter_itc else ""}
  {"<div class='card span-2'>" + safe_div(fig_itc_wg_share) + "</div>" if fig_itc_wg_share else ""}
</section>
"""

grid_html_wg = f"""
<section id="working-groups" class="grid">
  {"<div class='card'>" + safe_div(fig_wg_grouped)         + "</div>" if fig_wg_grouped else ""}
  {"<div class='card'>" + safe_div(fig_wg_100)             + "</div>" if fig_wg_100 else ""}
  {"<div class='card span-2'>" + safe_div(fig_wg_corr)     + "</div>" if fig_wg_corr else ""}
  {"<div class='card span-2 scroll-y'>" + safe_div(fig_country_wg_heat_all) + "</div>" if fig_country_wg_heat_all else ""}
</section>
"""

# ---- ALL DATA explorers (every country) ----
grid_html_all = f"""
<section id="all-data" class="grid">
  <div class="card scroll-y">{safe_div(fig_country_explorer)}</div>
  {"<div class='card scroll-y'>" + safe_div(fig_itc_share) + "</div>" if fig_itc_share else ""}
</section>
"""

# Full table (searchable)
table_full = (
    country_agg[
        ["Country","Total","MC","Core","Any WG","ITC participants","ITC_share","AnyWG_rate","ITC group"] + wg_cols
    ]
    .sort_values("Total", ascending=False)
    .rename(columns={"Any WG":"Any_WG","ITC participants":"ITC_participants"})
)
table_full_html = table_full.to_html(index=False, classes="datatable", border=0, table_id="countries_table")

page_html = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <title>Agrifood Evidence — Dashboard</title>
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  {PLOTLY_CDN}
  <style>
    :root {{
      --accent:{ACCENT}; --text:#111827; --muted:#6B7280; --bg:#ffffff; --card:#F8FAFC; --line:#E5E7EB;
      --itc:{ITC_GREEN}; --nonitc:{NONITC_GREY};
    }}
    * {{ box-sizing:border-box; }}
    html, body {{ margin:0; padding:0; background:var(--bg); color:var(--text); font-family: system-ui, -apple-system, Segoe UI, Roboto, Arial, sans-serif; }}
    a {{ color:var(--accent); text-decoration:none; }}
    .container {{ max-width: 1200px; margin: 1.25rem auto; padding: 0 1rem; }}
    header h1 {{ margin:0; font-size: 1.6rem; }}
    header .meta {{ color:var(--muted); margin:.25rem 0 1rem; }}
    .subnav {{
      position: sticky; top: 0; z-index: 50; backdrop-filter: blur(6px);
      background: rgba(255,255,255,0.75); border-bottom:1px solid var(--line);
      margin: .75rem 0 1rem; padding: .5rem .25rem;
    }}
    .subnav a {{ margin-right: 1rem; font-size: .95rem; color:#374151; }}
    .subnav a:hover {{ color:var(--accent); }}
    .kpis {{ display:grid; grid-template-columns: repeat(auto-fit,minmax(160px,1fr)); gap:12px; margin: 1rem 0 1.25rem; }}
    .kpi {{ background:var(--card); border:1px solid var(--line); border-radius:12px; padding:14px; }}
    .kpi-num {{ font-size:1.8rem; font-weight:700; line-height:1; }}
    .kpi-label {{ color:var(--muted); font-size:.95rem; }}
    .section-title {{ margin:1.25rem 0 .5rem; font-size:1.25rem; }}
    .grid {{ display:grid; gap:14px; grid-template-columns: repeat(auto-fit,minmax(320px,1fr)); margin-bottom: .5rem; }}
    .card {{ background:#fff; border:1px solid var(--line); border-radius:12px; padding:8px; min-height: 340px; }}
    .span-2 {{ grid-column: span 2; }}
    .scroll-y {{ max-height: 680px; overflow-y: auto; }}
    .datatable {{ border-collapse:collapse; width:100%; font-size: 0.95rem; }}
    .datatable th, .datatable td {{ border-bottom:1px solid var(--line); padding:6px 8px; text-align:left; }}
    .downloads {{ margin:.5rem 0 1rem; }}
    .search {{ margin:.5rem 0 .75rem; }}
    .search input {{ width:100%; padding:.5rem .6rem; border:1px solid var(--line); border-radius:8px; }}
    .badge {{ display:inline-block; padding:.15rem .4rem; border-radius:6px; font-size:.75rem; border:1px solid var(--line); color:#374151; }}
    .badge.itc {{ color:var(--itc); border-color:var(--itc); }}
  </style>
</head>
<body>
  <main class="container">
    <header>
      <h1>Agrifood Evidence — Dashboard</h1>
      <p class="meta">Interactive charts from <code>data/processed/data_clean.csv</code></p>
    </header>

    <nav class="subnav">
      <a href="#overview">Overview</a>
      <a href="#itc">ITC focus</a>
      <a href="#working-groups">Working Groups</a>
      <a href="#all-data">All Data</a>
      <a href="#table">Table</a>
    </nav>

    <section class="kpis">
      <div class="kpi"><div class="kpi-num">{kpi_total_people:,}</div><div class="kpi-label">Total people</div></div>
      <div class="kpi"><div class="kpi-num">{kpi_countries:,}</div><div class="kpi-label">Countries</div></div>
      <div class="kpi"><div class="kpi-num">{kpi_mc_total:,}</div><div class="kpi-label">MC members</div></div>
      <div class="kpi"><div class="kpi-num">{kpi_core_total:,}</div><div class="kpi-label">Core group</div></div>
      <div class="kpi"><div class="kpi-num">{kpi_any_wg_total:,}</div><div class="kpi-label">Any WG</div></div>
      {"<div class='kpi'><div class='kpi-num'>"+f"{kpi_itc_people:,}"+"</div><div class='kpi-label'>ITC participants</div></div>" if have_itc else ""}
      {"<div class='kpi'><div class='kpi-num'>"+f"{kpi_itc_countries:,}"+"</div><div class='kpi-label'>ITC countries represented</div></div>" if have_itc else ""}
    </section>

    <h2 class="section-title" id="overview">Overview</h2>
    {grid_html_overview}

    {"<h2 class='section-title' id='itc'>ITC focus <span class='badge itc'>ITC</span></h2>" if have_itc else ""}
    {grid_html_itc if have_itc else ""}

    <h2 class="section-title" id="working-groups">Working Groups</h2>
    {grid_html_wg}

    <h2 class="section-title" id="all-data">All Data (every country)</h2>
    {grid_html_all}

    <section id="table">
      <h2 class="section-title">Country table — all countries</h2>
      <div class="downloads">Download data: {downloads_html}</div>
      <div class="search">
        <input id="tableSearch" type="text" placeholder="Type to filter countries, e.g. 'poland' or 'itc'…" oninput="filterTable()" />
      </div>
      {table_full_html}
    </section>
  </main>

  <script>
    // simple client-side table filter
    function filterTable() {{
      const input = document.getElementById('tableSearch');
      const filter = input.value.toLowerCase();
      const table = document.getElementById('countries_table');
      const trs = table.getElementsByTagName('tr');
      for (let i = 1; i < trs.length; i++) {{
        const rowText = trs[i].innerText.toLowerCase();
        trs[i].style.display = rowText.indexOf(filter) > -1 ? '' : 'none';
      }}
    }}
  </script>
</body>
</html>
"""

out_path = DOCS_DIR / "dashboard.html"
out_path.write_text(page_html, encoding="utf-8")
print("Saved →", out_path.relative_to(ROOT))


Repo root → C:\Users\James\Documents\GitHub\evidence-map-agrifood
Saved → outputs\dashboard_country_agg.csv
Saved → docs\downloads\country_agg.csv
Saved → outputs\dashboard_wg_totals.csv
Saved → docs\downloads\wg_totals.csv
Saved → outputs\itc_summary.csv
Saved → docs\downloads\itc_summary.csv
Saved → docs\dashboard.html
